<a href="https://colab.research.google.com/github/suder54ul/LLMP/blob/main/module08_handson1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 08 - Hands-on 1: LLM Monitoring with LangSmith
**Large Language Models in Production**
**Open – May 2026**

**Facilitator:** A/P TAN Wee Kek  
**Email:** distwk@nus.edu.sg

**Learning Objectives** (from Module 7)
- Understand observability and monitoring in production LLM systems
- Use LangSmith to trace, measure, and analyse LLM performance
- Monitor latency, token usage, and cost in real time
- Apply monitoring insights to engineering for performance, efficiency, and change

## 🔧 LangSmith Setup Instructions (Required for this exercise)

**What is LangSmith Monitoring?**  
LangSmith is LangChain’s full observability platform. In this practical you will:
- Automatically trace every LLM call, prompt, and chain
- Monitor latency, token consumption, and estimated cost
- Explore detailed traces in the LangSmith dashboard
- Use monitoring data to support performance & efficiency decisions (Module 7)

**How to enable LangSmith (already done in previous exercise):**
1. Your LangSmith API key should already be in the SETUP cell below.
2. Make sure `LANGSMITH_TRACING = "true"` is active.
3. After running the notebook, go to [https://smith.langchain.com/](https://smith.langchain.com/) → **Monitoring** → `NUS_LLMP_Monitoring_Exercise`

**Note:** This notebook builds directly on the Evaluation practical. You will reuse the same query/document for easy comparison.

In [ ]:
# === FORCE DISABLE LANGSMITH TRACING (Run this cell first) ===
# import os

# # Explicitly disable tracing
# os.environ["LANGCHAIN_TRACING_V2"] = "false"

# # Remove any leftover LangSmith keys if they exist
# for key in list(os.environ.keys()):
#     if key.startswith("LANGCHAIN_"):
#         del os.environ[key]

# print("LangSmith tracing has been explicitly disabled.")

In [ ]:
# === SETUP (run first) ===
import os
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# === LANG SMITH MONITORING ENABLED ===
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "ur_langsmith_api_key_here"   # ← PASTE YOUR KEY HERE
os.environ["LANGSMITH_PROJECT"] = "NUS_LLMP_Evaluation_Exercise"

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0.0,
    num_ctx=8192,
)

print("Setup complete – LLaMA 3.1 8B + LangSmith Monitoring ENABLED!")

## Part 1: Run a Monitored LLM Call

In [ ]:
query = "What are the key risks of using AI in Singapore financial services?"

# Reuse the same MAS document from Evaluation notebook
document = """Excerpt from: Monetary Authority of Singapore (MAS) – Guidelines on Responsible Use of Generative AI in Financial Services (February 2026)

Financial institutions deploying generative AI systems must adopt a risk-based approach. The key risks identified include:

• Data Privacy and PDPA Compliance: Generative AI models may inadvertently memorise, reproduce or expose personal data. All inputs and outputs must be screened for personally identifiable information.

• Bias and Fairness: Models can perpetuate or amplify biases present in training data, leading to discriminatory outcomes in credit scoring, customer segmentation or insurance pricing.

• Explainability and Transparency: The “black-box” nature of large language models makes it difficult to explain decisions to customers and regulators. Institutions must maintain audit trails and be able to provide meaningful explanations.

• Cybersecurity and Robustness: AI systems are vulnerable to prompt injection, data poisoning and adversarial attacks. Robust input validation and continuous monitoring are required.

• Regulatory Compliance and Hallucinations: Models may generate plausible but factually incorrect information, potentially leading to non-compliance with MAS regulations or misleading customers.

• Human Oversight: High-risk use cases (e.g., credit decisions, compliance checks) must incorporate meaningful human-in-the-loop controls.

Institutions are expected to conduct thorough risk assessments, implement appropriate guardrails, and maintain comprehensive documentation for supervisory review."""

reference = """The key risks of using AI in Singapore financial services are data privacy/PDPA compliance, model bias and fairness, lack of explainability, cybersecurity vulnerabilities, hallucinations leading to regulatory non-compliance, and the need for human oversight in high-risk decisions."""

prompt = ChatPromptTemplate.from_template("Answer the question based on the document:\n\n{doc}\n\nQuestion: {q}")
chain = prompt | llm | StrOutputParser()

response = chain.invoke({"doc": document, "q": query})
print("=== MONITORED RESPONSE ===\n")
print(response)

## Part 2: View Your Traces in LangSmith Dashboard

1. Go to [https://smith.langchain.com/](https://smith.langchain.com/)
2. Open Monitoring **`NUS_LLMP_Monitoring_Exercise`**
3. Click on the latest run
4. Explore:
   - Full prompt & response
   - **Latency** (how long the call took)
   - **Token usage** (input + output tokens)
   - **Cost** estimation (even for local models)
   - Trace tree (prompt → LLM → parser)

## Part 3: Monitoring Multiple Runs & Performance Insights

In [ ]:
import time

print("Running 5 monitored calls for performance comparison...\n")
for i in range(5):
    start = time.time()
    resp = chain.invoke({"doc": document, "q": query})
    duration = time.time() - start
    print(f"Run {i+1}: {duration:.2f} seconds")
print("\nGo back to LangSmith to compare all 5 traces side-by-side!")

## Part 4–6: Your Turn + Group Discussion

**Exercise:**
1. Modify the temperature or add a tool call and re-run.
2. Observe how it affects latency and token usage in LangSmith.
3. Discuss: How would you use these monitoring insights to improve performance, efficiency, or manage change in a real production system?

**Key Takeaway:** Monitoring with LangSmith turns invisible LLM behaviour into actionable data — essential for engineering high-performance, efficient, and maintainable production systems.